In [ ]:
import os
import cv2
import tqdm
import torch
import numpy as np
from ultralytics import YOLO

In [ ]:
def countFrames(video):
    cap = cv2.VideoCapture(video)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    return frames

In [ ]:
def collectVideos(root):
    videos = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.upper().endswith(".MOV"):
                videos.append(os.path.join(dirpath, f))
    return videos

In [ ]:
def identifyPitcher(keypoints, frame_shape):
    if len(keypoints) == 1:
        return 0
    h, w = frame_shape[:2]
    expected_x = 0.5 * w
    idx, score = 0, float("-inf")

    for i, pts in enumerate(keypoints):
        xs, ys = pts[:, 0], pts[:, 1]
        x_diff = torch.max(pts[:, 0]) - torch.min(pts[:, 0])
        y_diff = torch.max(pts[:, 1]) - torch.min(pts[:, 1])
        hori = 1 - torch.abs(torch.nanmean(xs) - expected_x) / w
        if (s := 10 * x_diff * y_diff + hori) > score:
            idx, score = i, s
    return idx

In [ ]:
def processVideo(model, video, savedir):
    cap = cv2.VideoCapture(video)
    frame_idx, kp_seq = 0, []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        res = model(frame, verbose=False)[0]
        if not (res.keypoints and len(persons := res.keypoints.xy)):
            print(f"[WARN] No person detected at frame {frame_idx}")
            frame_idx += 1
            continue
        idx = identifyPitcher(persons, frame.shape)
        kp_seq.append(persons[idx].cpu().numpy())
        frame_idx += 1

    cap.release()
    if not kp_seq:
        print("[ERR] No keypoints extracted")
        return
    os.makedirs(savedir, exist_ok=True)
    base = os.path.splitext(os.path.basename(video))[0]
    np.save(os.path.join(savedir, f"{base}.npy"), np.array(kp_seq))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

model = YOLO("yolov8x-pose.pt")
if torch.cuda.is_available():
    model.to("cuda")

videos = collectVideos("/content/drive/My Drive/Pitch-Tipping/Pitches")
print(f"Found {len(videos)} video(s) to process.")

In [ ]:
%%time
for video in tqdm.tqdm(videos, desc="Processing"):
    print(f"\n--- Processing: {video} ---")
    processVideo(model, video, "/content/drive/My Drive/Pitch-Tipping/Keypoints")